# Model Evaluation

Runs NER, tags, and summary on 2 fixed eval docs for every model in `MODELS`.
Output is displayed inline for qualitative grading. Results are saved per model under `data/results/<model>/`.

In [1]:
MODELS = [
    # 'qwen2.5:7b',
    # 'llama3.2:latest',
    # 'qwen3:14b',
    'llama3.1:8b',
    # 'qwen3:8b',
]

EVAL_DOCS = [
    'medium_climate_blog_paris_agreement_en.md',
    'medium_klimaat_blog_parijs_akkoord_nl.md',
]

In [2]:
%load_ext autoreload
%autoreload 2

import json
import sys
import time
sys.path.insert(0, '../src')

from pathlib import Path

from dotenv import load_dotenv

from notemine.config import load_config
from notemine.frontmatter_io import load_note
from notemine.tasks import run_ner, run_summary, run_tags

load_dotenv('../.env')
config = load_config('../config.toml')
config['vault']['prompts_dir'] = '../' + config['vault']['prompts_dir'].lstrip('./')

EVAL_DOCS_DIR = Path('../data/synthetic_docs')
backend = config['run']['backend']
print(f'backend={backend}  models={MODELS}')

backend=ollama  models=['llama3.1:8b']


In [3]:
all_results = {}

for model in MODELS:
    model_safe = model.replace(':', '_')
    config[backend]['model'] = model
    print(f'\n--- {model} ---')
    results = []
    for doc in EVAL_DOCS:
        post = load_note(str(EVAL_DOCS_DIR / doc))
        row = {'file': doc}
        for task, fn in [('ner', run_ner), ('tags', run_tags), ('summary', run_summary)]:
            t0 = time.perf_counter()
            try:
                row[task] = fn(backend, post.content, config)
            except Exception as e:
                row[task] = f'ERROR: {e}'
            row[f'{task}_s'] = round(time.perf_counter() - t0, 1)
            print(f'  {doc}  {task}  ({row[f"{task}_s"]}s)')
        results.append(row)

    for task in ('ner', 'tags', 'summary'):
        out_dir = Path(f'../data/results/{model_safe}/{task}')
        out_dir.mkdir(parents=True, exist_ok=True)
        for r in results:
            stem = Path(r['file']).stem
            payload = r[task]
            is_error = isinstance(payload, str) and payload.startswith('ERROR:')
            data = {
                'file': r['file'],
                'backend': backend,
                'model': model,
                'elapsed_s': r[f'{task}_s'],
                task: payload if not is_error else [],
                'error': payload if is_error else None,
            }
            (out_dir / f'{stem}.json').write_text(json.dumps(data, ensure_ascii=False, indent=2))
    print(f'  saved to data/results/{model_safe}/')
    all_results[model] = results

print('\nAll models done')


--- llama3.1:8b ---
  medium_climate_blog_paris_agreement_en.md  ner  (26.2s)
  medium_climate_blog_paris_agreement_en.md  tags  (4.6s)
  medium_climate_blog_paris_agreement_en.md  summary  (6.1s)
  medium_klimaat_blog_parijs_akkoord_nl.md  ner  (26.8s)
  medium_klimaat_blog_parijs_akkoord_nl.md  tags  (5.2s)
  medium_klimaat_blog_parijs_akkoord_nl.md  summary  (6.2s)
  saved to data/results/llama3.1_8b/

All models done


In [4]:
for model, results in all_results.items():
    print(f"\n{'#'*60}")
    print(f"MODEL: {model}")
    for r in results:
        print(f"\n{'='*60}")
        print(f"DOC: {r['file']}\n")
        print(f"NER ({r['ner_s']}s):")
        print(json.dumps(r['ner'], ensure_ascii=False, indent=2))
        print(f"\nTAGS ({r['tags_s']}s):")
        print(json.dumps(r['tags'], ensure_ascii=False, indent=2))
        print(f"\nSUMMARY ({r['summary_s']}s):")
        print(json.dumps(r['summary'], ensure_ascii=False, indent=2))


############################################################
MODEL: llama3.1:8b

DOC: medium_climate_blog_paris_agreement_en.md

NER (26.2s):
[
  [
    "Le Bourget",
    "LOC"
  ],
  [
    "France",
    "LOC"
  ],
  [
    "Paris Agreement",
    "PROD"
  ],
  [
    "December 12, 2015",
    "DATE"
  ],
  [
    "Dr. Friederike Otto",
    "PER"
  ],
  [
    "Grantham Institute",
    "ORG"
  ],
  [
    "Imperial College London",
    "ORG"
  ],
  [
    "Professor Pierre Friedlingstein",
    "PER"
  ],
  [
    "University of Exeter",
    "ORG"
  ],
  [
    "Global Carbon Project",
    "PROD"
  ],
  [
    "International Energy Agency",
    "ORG"
  ],
  [
    "Paris",
    "LOC"
  ],
  [
    "South and Southeast Asia",
    "LOC"
  ],
  [
    "European Union",
    "ORG"
  ],
  [
    "Ursula von der Leyen",
    "PER"
  ],
  [
    "European Green Deal",
    "MISC"
  ],
  [
    "Brazil",
    "LOC"
  ],
  [
    "President Luiz Inácio Lula da Silva",
    "PER"
  ],
  [
    "Amazon Basin",
    "LOC"
 